In [1]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient
import pandas as pd

## Connect to DB and import data

We can use the same process as we did in app.py and import the logs from our mongodb cluster.

In [2]:
db_name = "vancrime"
collection_name = "query_log"

In [3]:
# Load all documents from a MongoDB collection into a Pandas DataFrame.

load_dotenv()
uri = os.getenv("PYMONGO_URI")

if uri is None:
    raise ValueError("PYMONGO_URI not found in .env file")

client = MongoClient(uri)

db = client[db_name]
collection = db[collection_name]

docs = list(collection.find())
df = pd.DataFrame(docs)

if "_id" in df.columns:
    df["_id"] = df["_id"].astype(str)

In [4]:
df.head()

,_id,user_query,sql,tool,llm_output,input_tokens,output_tokens,cache_read_tokens,cost,timestamp,model,n_rows,session_id,user_token,cache_write_tokens
0,69b1d3232d6a85ba1b0c35b7,top 5 crime in kerrisdale,"\nSELECT \n TYPE,\n COUNT(*) AS count\nFROM ...",querychat_query,The top 5 crimes in Kerrisdale are:\n\n| Crime...,288,227,4332,0.001856,2026-03-11T13:40:03,anthropic/claude-haiku-4-5,102874,aac2368c694a86894406be4831c91ecc58ed3f5a794b2d...,vc-a22b45df-87bb-48b8-adc1-498597239abe,NaN
1,69b1d3b12d6a85ba1b0c35b8,See crime trends in Kerrisdale over time (2023...,"\nSELECT \n YEAR,\n MONTH,\n COUNT(*) AS to...",querychat_query,,314,149,4559,0.001515,2026-03-11T13:42:25,anthropic/claude-haiku-4-5,102874,aac2368c694a86894406be4831c91ecc58ed3f5a794b2d...,vc-a22b45df-87bb-48b8-adc1-498597239abe,NaN
2,69b1d47152a4395e40daf026,crime in and near downtown,SELECT * FROM vancouver_crime WHERE NEIGHBOURH...,querychat_update_dashboard,I've filtered the dashboard to show crime in D...,4,130,4328,0.000000,2026-03-11T13:45:37,anthropic/claude-haiku-4-5,34692,be2976246a0dd5d1349aded0d749ea9a028ad74bf2dcf5...,vc-a22b45df-87bb-48b8-adc1-498597239abe,121.0
3,69b1d765ad68ed2dd9cab8b0,Filter to crimes in Downtown Vancouver,SELECT * FROM vancouver_crime WHERE NEIGHBOURH...,querychat_update_dashboard,Hello! I'm your data dashboard chatbot for the...,2,190,0,0.006359,2026-03-11T13:58:13,anthropic/claude-haiku-4-5,34692,6d0624f5c6d8e0036405b0ec7431a4e3222ada575e0cce...,vc-a22b45df-87bb-48b8-adc1-498597239abe,4326.0
4,69b1d778ad68ed2dd9cab8b1,crims in hastingsg,SELECT * FROM vancouver_crime WHERE NEIGHBOURH...,querychat_update_dashboard,Done! I've filtered the dashboard to show crim...,3,121,4771,0.001241,2026-03-11T13:58:32,anthropic/claude-haiku-4-5,2751,6d0624f5c6d8e0036405b0ec7431a4e3222ada575e0cce...,vc-a22b45df-87bb-48b8-adc1-498597239abe,125.0


## Description of Columns

* **_id**: Unique identifier for each log entry (MongoDB document ID).
* **user_query**: The natural language question or request submitted by the user.
    - in cases where LLM makes incorrect queries or re-queries the data it is seen that the `user_query` field is sometimes empty. the missing info can be tracked down by identifying session-id and user-id and associated with the queries made on that session
* **sql**: The SQL query generated (or used) by the querychat to answer the user’s request.
* **tool**: The internal tool or function invoked (for this log we are only noting down two: `querychat_update_dashboard` and `querychat_query`).
    - only `querychat_update_dashboard` is expected to the dataframe and thus show outputs. Thus nrow for `querychat_query` is expected to be the same every time
* **llm_output**: The response generated by the language model and shown to the user.
    - in some cases the llm output is not sent to the api callback properly, in which case the previous message or an empty string is sometimes recorded. these need to be taken care of before processing
* **timestamp**: Time when the interaction occurred
* **model**: The specific LLM used to generate the response (currently we use only one but it can be used to compare more types of llm models if required).
* **n_rows**: Number of rows returned from the query result. in case the llm has not filtered any dataframe all rows are returned.
* **session_id**: Identifier grouping interactions within the same user session.
* **user_token**: Unique identifier for the user. it is a unique cookie generated for each user and stored in their local browser. it can be used to track users more efficiently than session-ids without having the user to log in. the downside is that the same user cannot retain their history over different browsers and is prone to getting deleted if cache is cleared. for our purposes we can treat each user_token as a single user.
* **input_tokens**: Number of tokens sent to the model as input.
* **output_tokens**: Number of tokens generated by the model in response.
* **cache_read_tokens**: Tokens retrieved from cache (reducing compute cost).
* **cost**: Estimated monetary cost of the model call in USD.
* **cache_write_tokens**: Tokens written to cache for future reuse.


#### Tool calling

1. we can check which tool is being called most frequently

In [5]:
tool_usage = df["tool"].value_counts()
tool_usage

tool
querychat_update_dashboard    26
querychat_query               17
Name: count, dtype: int64

2. we can check token and cost usage for different tokens

In [6]:
tool_stats = (
    df.groupby("tool")
      .agg(
          n_calls=("tool", "count"),
          avg_cost=("cost", "mean"),
          total_cost=("cost", "sum"),
          avg_input_tokens=("input_tokens", "mean"),
          avg_output_tokens=("output_tokens", "mean"),
          avg_cache_read=("cache_read_tokens", "mean"),
          avg_cache_write=("cache_write_tokens", "mean"),
      )
      .sort_values("total_cost", ascending=False)
)

tool_stats

,n_calls,avg_cost,total_cost,avg_input_tokens,avg_output_tokens,avg_cache_read,avg_cache_write
tool,,,,,,,
querychat_update_dashboard,26,0.001751,0.045531,3.192308,130.230769,4983.230769,517.153846
querychat_query,17,0.002267,0.038546,39.235294,257.529412,5256.000000,376.466667


In [7]:
df["active_tokens"] = df["input_tokens"] + df["output_tokens"]
df["cache_tokens"] = df["cache_read_tokens"] + df["cache_write_tokens"]

totals = {
    "total_active_tokens": df["active_tokens"].sum(),
    "total_cache_tokens": df["cache_tokens"].sum(),
    "cache_to_active_ratio": df["cache_tokens"].sum() / df["active_tokens"].sum()
}

pd.DataFrame([totals])

,total_active_tokens,total_cache_tokens,cache_to_active_ratio
0,8514,229118.0,26.910735


In [8]:
# Flag queries with poor cache utilization
df['cache_hit_rate'] = df['cache_read_tokens'] / (df['input_tokens'] + df['cache_read_tokens'].fillna(0))
df['timestamp'] = pd.to_datetime(df['timestamp'])

cache_efficiency = df[['timestamp', 'user_query', 'tool', 'cache_hit_rate', 'cost']]\
    .sort_values('cache_hit_rate')

# Bottom 5 least cache-efficient calls
cache_efficiency.head()

,timestamp,user_query,tool,cache_hit_rate,cost
3,2026-03-11 13:58:13,Filter to crimes in Downtown Vancouver,querychat_update_dashboard,0.000000,0.006359
8,2026-03-11 21:21:19,Show me all bike thefts in Kitsilano,querychat_update_dashboard,0.000000,0.006548
1,2026-03-11 13:42:25,See crime trends in Kerrisdale over time (2023...,querychat_query,0.935563,0.001515
0,2026-03-11 13:40:03,top 5 crime in kerrisdale,querychat_query,0.937662,0.001856
6,2026-03-11 14:12:15,,querychat_update_dashboard,0.998615,0.001159


This can help us check whether either tool is disproportionately being called or re-called, and whether it is consuming many tokens. We can compare the types of token usage. Low token usage could be counter acted by high tool calling, and we can analyse the behaviour using this column. we can also check cache utilisation and whether it is helping with cost.

### Session and user specific analysis

1. we can check how each session and each user is querying the chatbot. This can enable us to estimate how much the user is interacting with the AI part. We can proceed to check if the same user and session is having repeat queries, which could mean that the user is trying to get some specific information but is unable to get the exact output they desire. Or we could check if there is a meaningful process of querying where there are diverse queries being made one after the other. we would try to aim for some semantic similarity too because random querying also wont be very useful.

In [9]:
# Per session
session_repeat = df.groupby("session_id").size()
avg_session_repeats = session_repeat.mean()

# Per user
user_repeat = df.groupby("user_token").size()
avg_user_repeats = user_repeat.mean()

print({
    "avg_queries_per_session": avg_session_repeats,
    "avg_queries_per_user": avg_user_repeats
})

{'avg_queries_per_session': np.float64(2.263157894736842), 'avg_queries_per_user': np.float64(8.6)}


In [10]:
repeat_queries = (
    df.groupby(["session_id", "user_query"])
      .size()
      .reset_index(name="count")
)

repeats_only = repeat_queries[repeat_queries["count"] > 1]

print(repeats_only.sort_values("count", ascending=False).head(10))

                                          session_id user_query  count
5  37344e66b0d8fe2f142bd63468003b677b7f9403075421...                 2


In [11]:
session_stats = (
    df.groupby("session_id")
      .agg(
          n_queries=("user_query", "count"),
          total_cost=("cost", "sum"),
          avg_rows_returned=("n_rows", "mean")
      )
      .sort_values("n_queries", ascending=False)
)

session_stats.head(5)

,n_queries,total_cost,avg_rows_returned
session_id,,,
7fc79b8919e398ec31b53af5989a1e97059739e69a7679bb202e4cc5280c76bf,9,0.012679,1.777778
04497d7c9f3f643a394acbc5efc4cc3d549ba2d344677bb060f410c9f2de3cbb,5,0.014750,65934.800000
37344e66b0d8fe2f142bd63468003b677b7f94030754216a513abe733764c878,3,0.003718,102874.000000
4c58157c4c787a5ab712517a34a76ea2a841eb6ca46d534e837e0e959f1d00f3,2,0.003346,56404.500000
63f214934038733fa300dcbcd5e6ffa2d1796c8ce3a6b0027f8b70eadbba30ce,2,0.005975,6143.500000


### Query Analysis

- we can actually use LLMs or any other language processing tool to summarize the frequent queries being made.
- we can thus try to add them as the main suggestions, use that information to tweak our charts, add RAG or any context outline for specific 
- look for specific language eg. central business district is always being referred to as downtown so it could be a good idea to change the name directly. then we could check if it creates any confusion with adjacent area (eg. now people think downtown should naturally include West End too). This could be done regarding crime type too
- look for frequent user mistakes/clarification. 
- we can analyse the llm output to check if the llm is giving vague answers, is confident in giving wrong answers or asks user for clarification. we can add that info to its extra instructions too.

In [12]:
import re

def clean_query(q):
    q = q.lower()
    q = re.sub(r"[^a-z0-9\s]", "", q)
    return q.strip()

df["clean_query"] = df["user_query"].apply(clean_query)

top_queries = (
    df["clean_query"]
    .value_counts()
    .head(10)
)

top_queries

clean_query
                                                     4
show me all bike thefts in kitsilano                 2
top 5 crime in kerrisdale                            1
see crime trends in kerrisdale over time 20232025    1
crime in and near downtown                           1
filter to crimes in downtown vancouver               1
crims in hastingsg                                   1
test 1                                               1
show me bike thefts across all neighborhoods         1
which month is safest                                1
Name: count, dtype: int64

In [13]:
# Get top 10 queries
top_queries = top_queries.index.tolist()

# Build sample prompt
prompt = f"""
You are analyzing user behavior for a crime analytics chatbot.

Top 10 user queries:
{chr(10).join(f"- {q}" for q in top_queries)}

Tasks:
1. Identify main user intents
2. Highlight any issues (repetition, ambiguity, typos)
3. suggestions for improvements to UX or system responses
"""

print(prompt)


You are analyzing user behavior for a crime analytics chatbot.

Top 10 user queries:
- 
- show me all bike thefts in kitsilano
- top 5 crime in kerrisdale
- see crime trends in kerrisdale over time 20232025
- crime in and near downtown
- filter to crimes in downtown vancouver
- crims in hastingsg
- test 1
- show me bike thefts across all neighborhoods
- which month is safest

Tasks:
1. Identify main user intents
2. Highlight any issues (repetition, ambiguity, typos)
3. suggestions for improvements to UX or system responses



### Time and simultaneous queries analysis

- we can check whether there are multiple queries being made simultaneously. this could reach rate limit for our tokens so we can check time-specific user requirements
- we can do this globally, or even per-user specific

In [14]:
window='600s' #using 5 min window size since we do not have a lot of data
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Set timestamp as index
df_indexed = df.set_index("timestamp")

counts = df_indexed.resample('600s')["user_query"].count() 

pd.DataFrame(counts.head(5))

,user_query
timestamp,
2026-03-11 13:40:00,3
2026-03-11 13:50:00,2
2026-03-11 14:00:00,0
2026-03-11 14:10:00,3
2026-03-11 14:20:00,1
